# ReAct 에이전트 — 발표 자료
### Reasoning + Acting : AI가 "생각하고 행동하는" 방법

---

## 발표 순서

| 순서 | 내용 |
|------|------|
| 1 | ReAct가 무엇인지 — 비유로 이해하기 |
| 2 | 기존 방식과 무엇이 다른가 |
| 3 | 핵심 구조 : Thought → Action → Observation |
| 4 | 응용 예제 1 — 직접 만든 SimpleReactAgent (서울 인구밀도 계산) |
| 5 | 응용 예제 2 — Function Calling 방식 (날씨 비교 + 환율 계산) |
| 6 | 두 방식 비교 및 정리 |

---

> 코딩을 전혀 몰라도 괜찮습니다. 코드 설명은 모두 한국어 주석과 발표 대본으로 풀어드립니다.


---
## PART 1. ReAct가 뭔가요? — 탐정 비유로 이해하기

---

### 탐정과 AI는 닮았다

보통 AI에게 질문하면 **바로 답**합니다.  
그런데 가끔 자신 있게 **틀린 답**을 말해요. 이걸 **"환각(Hallucination)"** 이라고 합니다.

> 기존 방식: "서울 인구밀도가 얼마예요?" → AI가 기억 속에서 바로 답함 (틀릴 수도 있음)

ReAct는 다릅니다. 마치 **탐정처럼** 움직입니다.

```
탐정의 사건 해결 방식:
  "범인을 찾으려면 알리바이와 목격자 진술이 필요해"    <- Thought (생각)
  "목격자를 인터뷰하러 간다"                           <- Action (행동)
  "목격자: 그날 밤 그 사람 봤어요"                     <- Observation (관찰)
  "알리바이도 확인해야겠어"                            <- 다시 Thought
  "CCTV 영상을 확인한다"                              <- 다시 Action
  "증거 충분! 범인은 OOO"                             <- Finish (최종 답변)
```

**ReAct = 탐정처럼 생각하고, 도구를 써서 확인하고, 다시 생각하는 AI**

---

### 핵심 한 줄 요약

> **ReAct = Reasoning(추론) + Acting(행동)**  
> AI가 모르면 도구를 써서 확인하고, 확인한 뒤에 답하는 구조


---
## PART 2. 기존 방식과 무엇이 다른가?

---

### AI 프롬프팅 기법의 진화

```
Zero-shot  --> Few-shot  --> Chain-of-Thought  --> ReAct
  (즉답)       (예시줌)       (단계별 사고)        (도구 사용)
```

| 방법 | 설명 | 외부 도구 | 환각 감소 |
|------|------|-----------|-----------|
| Zero-shot | 아무 힌트 없이 바로 질문 | 없음 | 낮음 |
| Few-shot | 예시를 먼저 보여주고 질문 | 없음 | 중간 |
| Chain-of-Thought | "단계별로 생각해봐"라고 시킴 | 없음 | 중간 |
| **ReAct** | 생각 + 실제 도구 사용 | **있음** | **높음** |

---

### 핵심 차이: 도구를 쓸 수 있냐 없냐

```
기존 방식 (Chain-of-Thought):
  AI의 머릿속에서만 계산 -- 틀려도 그냥 출력됨

ReAct 방식:
  Thought: 인구와 면적을 조회해야 합니다
  Action: knowbs[서울 인구]          -- 실제 도구 실행
  Observation: 950만 명              -- 도구가 돌려준 실제 결과
  Action: knowbs[서울 면적]
  Observation: 605.2 km2
  Action: Calc[9500000/605.2]        -- 계산기 실제 실행
  Observation: 15697.29
  Action: Finish[인구밀도 약 15,697명/km2]
```


---
## PART 3. ReAct의 핵심 구조

---

### 루프 구조: Thought → Action → Observation → (반복) → Finish

```
질문 입력
   |
   v
[ Thought ]  : AI가 "지금 뭘 해야 하지?" 추론
   |
   v
[ Action  ]  : 도구 이름과 입력값 지정
   |
   v
[ Observation ] : 도구 실행 결과 확인
   |
   v
충분한 정보가 모였나?
  아니오 --> 다시 Thought로 돌아감
  예     --> Finish (최종 답변 제출)
```

---

### 반드시 지켜야 할 출력 형식

AI가 이 형식을 정확히 따라야 코드가 파싱(읽기)할 수 있습니다.

```
Thought: [지금 상황에 대한 추론]
Action: 도구이름[입력값]

(도구 실행 후 시스템이 자동으로 추가)
Observation: [도구 실행 결과]

...반복...

Thought: [최종 추론]
Action: Finish[최종 답변]
```

---

### 안전장치: 최대 반복 횟수 (max_steps)

AI가 루프를 무한 반복하면 API 비용이 폭발합니다.
그래서 `max_steps = 5` 같이 최대 몇 번까지만 반복할지 미리 정해둡니다.




---
## PART 4. 응용 예제 1 — SimpleReactAgent (서울 인구밀도 계산)

### 예제 개요
- 질문: "서울의 인구와 면적을 조회하고, 인구밀도를 계산해주세요"
- 도구: 지식베이스 검색(knowbs) + 계산기(Calc)
- 방식: 텍스트 파싱 기반 — AI 출력을 직접 읽어서 도구 호출

---

### 도구(Tool) 정의
> 도구란? AI가 스스로 할 수 없는 일을 대신해주는 함수입니다.
> 사람이 계산기 앱을 켜서 숫자를 입력하는 것처럼, AI는 코드로 이 함수들을 호출합니다.


In [4]:
# ===================================================================
# [도구 정의]
#
# 비전공자 설명:
#   도구 = AI가 직접 실행시키는 함수
#   사람이 계산기 앱을 쓰는 것처럼, AI는 이 함수들을 호출합니다.
# ===================================================================

def calculator(expression):
    # [도구 1] 계산기
    # 역할: 수학 수식 문자열을 받아서 계산 결과를 문자열로 반환
    # 예시: calculator("9500000/605.2") -> "15697.29..."
    # 보안: 숫자와 사칙연산 기호만 허용 (악의적 코드 실행 방지)
    allowed = set('0123456789+-*/.() ')
    if all(c in allowed for c in expression):
        return str(eval(expression))   # eval = 문자열 수식을 실제로 계산
    return "허용되지 않는 수식입니다."


def knowledge_base(query):
    # [도구 2] 지식베이스 검색
    # 역할: 미리 저장된 사전에서 키워드와 일치하는 정보를 반환
    # 예시: knowledge_base("서울 인구") -> "서울특별시의 인구는 약 950만 명입니다"
    #
    # 실제 서비스에서는 이 부분을 웹 검색 API나 데이터베이스 조회로 교체합니다.
    kb = {
        "서울 인구": "서울특별시의 인구는 약 950만 명입니다 (2024년 기준).",
        "서울 면적": "서울특별시의 면적은 약 605.2 km2 입니다.",
        "한국 GDP":  "대한민국의 GDP는 약 1조 7천억 달러입니다 (2024년 기준).",
        "한국 인구": "대한민국의 총 인구는 약 5,170만 명입니다 (2024년 기준).",
        "파이썬":    "Python은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다.",
        "인공지능":  "인공지능(AI)은 인간의 지능을 모방하는 컴퓨터 시스템을 연구하는 분야입니다.",
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower or all(w in query_lower for w in key.split()):
            return value
    return f"'{query}'에 대한 정보를 찾을 수 없습니다."


# 도구 동작 확인
print("도구 정의 완료")
print(f"  calculator('100+200')        -> {calculator('100+200')}")
print(f"  knowledge_base('서울 인구')  -> {knowledge_base('서울 인구')}")


도구 정의 완료
  calculator('100+200')        -> 300
  knowledge_base('서울 인구')  -> 서울특별시의 인구는 약 950만 명입니다 (2024년 기준).


---
### SimpleReactAgent 클래스

> 클래스란? 관련된 기능들을 하나로 묶은 설계도입니다.
> 에이전트 설계도를 만들어두면, 도구만 바꿔서 다양한 용도의 에이전트를 만들 수 있습니다.


In [5]:
import re
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


class SimpleReactAgent:
    # ================================================================
    # 비전공자 설명 -- SimpleReactAgent 전체 흐름
    #
    # 1. register_tool()         -> 도구를 에이전트에 등록 (레고 블록 추가)
    # 2. _build_system_prompt()  -> AI에게 "넌 이런 도구를 쓸 수 있고,
    #                               이런 형식으로 답해야 해"라는 지시문 작성
    # 3. run(question)           -> 실제 루프 실행
    #    AI 호출 -> 출력 파싱 -> 도구 이름 추출 -> 도구 실행 -> 결과를 AI에게 전달
    #    위 과정을 Finish 나올 때까지 반복
    # ================================================================

    def __init__(self, model='gpt-4o-mini', max_steps=5):
        self.model = model
        self.max_steps = max_steps  # 무한 루프 방지용 최대 반복 횟수
        self.tools = {}             # 등록된 도구들을 담는 딕셔너리
        self.system_prompt = ''

    def register_tool(self, name, func, description):
        # 도구를 에이전트에 추가합니다.
        #
        # name        : AI가 호출할 때 쓸 이름 (예: 'Calc')
        # func        : 실제로 실행할 파이썬 함수
        # description : AI에게 이 도구의 사용법을 알려주는 설명문
        #               -> 설명이 명확할수록 AI가 올바른 도구를 선택합니다
        self.tools[name] = {'func': func, 'description': description}

    def _build_system_prompt(self):
        # AI에게 전달하는 '역할 지시문'을 자동으로 생성합니다.
        #
        # 포함 내용:
        #   (1) 에이전트의 역할 설명
        #   (2) 사용 가능한 도구 목록
        #   (3) 반드시 지켜야 할 출력 형식 (Thought/Action/Observation)
        #
        # AI는 이 지시문을 기반으로 어떤 도구를 언제 쓸지 판단합니다.
        tool_desc = '\n'.join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])
        self.system_prompt = f'''당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.

사용 가능한 도구:
{tool_desc}

반드시 다음 형식을 따라 응답하세요:
Thought: [현재 상황에 대한 추론]
Action: [도구이름][입력값]

도구 실행결과는 Observation으로 제공됩니다.
최종 답변을 제출할 때는 반드시 다음 형식을 사용하세요:
Thought: [최종 추론]
Action: Finish[최종 답변]

중요: 한 번에 하나의 Action만 수행하세요.
'''

    def _parse_action(self, text):
        # AI가 출력한 텍스트에서 도구 이름과 입력값을 추출합니다.
        #
        # 정규표현식 설명:
        #   r'Action\s*:\s*(\w+)\[(.+?)\]'
        #   "Action: 도구이름[입력값]" 패턴을 찾아서
        #   도구이름과 입력값을 각각 뽑아냄
        #
        # 예시: "Action: Calc[9500000/605.2]"
        #        -> 도구이름: "Calc", 입력값: "9500000/605.2"
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None

    def _execute_tool(self, tool_name, tool_input):
        # 파싱된 도구 이름과 입력값으로 실제 함수를 실행합니다.
        # Finish는 "종료 신호"이므로 실행하지 않고 None 반환.
        if tool_name == 'Finish':
            return None
        elif tool_name in self.tools:
            try:
                return self.tools[tool_name]['func'](tool_input)
            except Exception as e:
                return f'도구 실행 오류: {e}'
        else:
            return f"'{tool_name}' 도구를 찾을 수 없습니다."

    def run(self, question):
        # ReAct 루프의 핵심 메서드입니다.
        #
        # 루프 흐름:
        #   1. AI에게 메시지 전달 -> AI 출력(Thought+Action) 수신
        #   2. 출력을 파싱해서 도구 이름과 입력값 추출
        #   3. "Finish"이면 루프 종료, 그 외에는 도구 실행
        #   4. 도구 실행 결과(Observation)를 다시 메시지에 추가
        #   5. max_steps 도달 전까지 반복
        self._build_system_prompt()

        # 대화 내용을 담는 리스트 -- AI와 주고받은 전체 내용을 누적
        messages = [
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user',   'content': f'Question: {question}'}
        ]

        print(f'\n질문: {question}')
        print('=' * 60)

        for step in range(self.max_steps):
            # 1. AI 호출
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0    # 0 = 가장 결정론적 (같은 입력이면 같은 출력)
            )
            assistant_msg = response.choices[0].message.content
            messages.append({'role': 'assistant', 'content': assistant_msg})

            print(f'\n[Step {step+1}]')
            print(assistant_msg)

            # 2. AI 출력 파싱
            action_name, action_input = self._parse_action(assistant_msg)

            # 3. 종료 조건 확인
            if action_name == 'Finish':
                print(f'\n{"=" * 60}')
                print(f'최종 답변: {action_input}')
                return action_input

            # 4. 도구 실행 및 결과 추가
            elif action_name:
                observation = self._execute_tool(action_name, action_input)
                obs_msg = f'Observation: {observation}'
                messages.append({'role': 'user', 'content': obs_msg})
                print(f'\n{obs_msg}')

            # 5. 형식 오류 처리
            else:
                messages.append({
                    'role': 'user',
                    'content': '형식 오류: Action: 도구이름[입력값] 형식으로 응답해주세요'
                })

        print('\n최대 단계 수에 도달했습니다.')
        return None


---
### 실제 실행 — 서울 인구밀도 계산

> 아래 셀을 실행하면 AI가 스스로 단계를 밟아 답을 구합니다.


In [6]:
# ===================================================================
# [에이전트 생성 및 실행]
#
# 비전공자 설명:
#   1. 에이전트 생성 (설계도에서 실제 객체 만들기)
#   2. 도구 2개 등록 (Calc, knowbs)
#   3. 질문 입력 -> AI가 자동으로 루프 실행
# ===================================================================

# (1) 에이전트 생성
agent = SimpleReactAgent(model='gpt-4o-mini', max_steps=5)

# (2) 도구 등록 (레고 블록 추가하듯 필요한 도구를 붙임)
agent.register_tool(
    'Calc',
    calculator,
    '수학 수식을 계산합니다. 예: Calc[2+3], Calc[9500000/605.2]'
)
agent.register_tool(
    'knowbs',
    knowledge_base,
    '내장 지식베이스에서 정보를 검색합니다. 예: knowbs[서울 인구], knowbs[한국 GDP]'
)

# (3) 질문 실행 -- AI가 Thought->Action->Observation 루프를 자동으로 돌림
result = agent.run('서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)를 계산해주세요')



질문: 서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)를 계산해주세요

[Step 1]
Thought: 서울의 인구와 면적을 조회하여 인구밀도를 계산해야 합니다. 먼저 서울의 인구를 조회하겠습니다. 
Action: knowbs[서울 인구]

Observation: 서울특별시의 인구는 약 950만 명입니다 (2024년 기준).

[Step 2]
Thought: 서울의 인구는 약 950만 명임을 확인했습니다. 다음으로 서울의 면적을 조회하겠습니다. 
Action: knowbs[서울 면적]

Observation: 서울특별시의 면적은 약 605.2 km2 입니다.

[Step 3]
Thought: 서울의 인구는 약 950만 명이고, 면적은 약 605.2 km²입니다. 이제 인구밀도를 계산하겠습니다. 인구밀도는 인구를 면적으로 나누어 계산합니다. 
Action: Calc[9500000/605.2]

Observation: 15697.290152015861

[Step 4]
Thought: 서울의 인구밀도는 약 15,697.29명/km²입니다. 
Action: Finish[서울의 인구밀도는 약 15,697.29명/km²입니다.]

최종 답변: 서울의 인구밀도는 약 15,697.29명/km²입니다.


---
### 실행 결과 해석

```
[Step 1]
Thought: 인구와 면적을 먼저 조회해야 합니다.
Action: knowbs[서울 인구 면적]
Observation: 서울특별시의 인구는 약 950만 명입니다.   -- 지식베이스가 돌려줌

[Step 2]
Thought: 면적을 추가로 확인해야 합니다.
Action: knowbs[서울 면적]
Observation: 서울특별시의 면적은 약 605.2 km2 입니다.

[Step 3]
Thought: 두 값을 이용해 인구밀도를 계산합니다.
Action: Calc[9500000/605.2]
Observation: 15697.29...                               -- 계산기가 실제로 계산

[Step 4]
Action: Finish[인구밀도 약 15,697명/km2]              -- 루프 종료!
```

AI가 스스로 어떤 도구를 언제 써야 할지 판단했습니다.
개발자가 "2번째엔 면적 조회해"라고 따로 알려주지 않았습니다.


---
## PART 5. 응용 예제 2 — Function Calling (날씨 비교 + 환율 계산)

### 예제 개요
- 질문 A: "서울과 도쿄의 현재 날씨를 비교하고, 온도 차이를 계산해주세요"
- 질문 B: "현재 1달러는 몇 원인지 조회하고, 220달러를 원화로 환산해주세요"
- 도구: 날씨 API + 실시간 환율 API + 계산기
- 방식: Function Calling -- AI가 JSON 형태로 구조화된 도구 호출

---

### 텍스트 파싱 방식 vs Function Calling 방식

| 구분 | 텍스트 파싱 (예제 1) | Function Calling (예제 2) |
|------|----------------------|---------------------------|
| AI 출력 형태 | "Action: Calc[100+200]" (자유 텍스트) | {"name":"calculate","args":{...}} (JSON) |
| 파싱 필요 여부 | 정규표현식으로 직접 파싱 | 파싱 불필요, 구조가 보장됨 |
| 안정성 | AI가 형식을 틀릴 수도 있음 | JSON 스키마로 타입 검증 |
| 병렬 호출 | 한 번에 하나씩만 | 여러 도구를 동시에 호출 가능 |

Function Calling은 OpenAI가 제공하는 기능으로, AI가 텍스트 대신 정해진 JSON 형식으로 도구 호출 정보를 반환합니다.

---

### 도구와 에이전트 설정


In [7]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# ===================================================================
# 비전공자 설명 -- JSON Schema = 도구 사용 설명서
#
# AI에게 도구를 알려줄 때, 그냥 설명문이 아니라
# "이 도구는 city라는 문자열 인자를 받아요"처럼
# 정형화된 JSON 형식으로 알려줍니다.
# -> AI가 정확한 형태로 도구를 호출할 수 있게 됩니다.
# ===================================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 실제 API로 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름 (예: Seoul, Tokyo)"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "온도 단위 (기본값: celsius)"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수학 계산을 수행합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 수식 (예: 25 - 22)"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "두 통화 간의 실시간 환율을 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_currency": {
                        "type": "string",
                        "description": "기준 통화 (예: USD, KRW, JPY)"
                    },
                    "target_currency": {
                        "type": "string",
                        "description": "대상 통화 (예: USD, KRW, JPY)"
                    }
                },
                "required": ["base_currency", "target_currency"]
            }
        }
    }
]


def get_weather(city, unit='celsius'):
    # wttr.in 공개 API로 실제 날씨를 조회합니다.
    # 비전공자 설명: 인터넷에 있는 날씨 서비스에 직접 요청해서 데이터를 받아옵니다.
    try:
        url = f'https://wttr.in/{city}?format=j1'
        resp = requests.get(url, timeout=10)
        data = resp.json()
        current = data['current_condition'][0]
        temp = current['temp_C'] if unit == 'celsius' else current['temp_F']
        unit_str = 'C' if unit == 'celsius' else 'F'
        return json.dumps({
            'city': city,
            'temperature': f'{temp} degrees {unit_str}',
            'description': current['weatherDesc'][0]['value'],
            'humidity': f"{current['humidity']}%"
        }, ensure_ascii=False)
    except Exception as e:
        return json.dumps({'city': city, 'error': str(e)})


def calculate(expression):
    # 수학 수식을 계산합니다.
    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": str(result)})
    except Exception as e:
        return json.dumps({"error": str(e)})


def get_exchange_rate(base_currency, target_currency):
    # open.er-api.com 공개 API로 실시간 환율을 조회합니다.
    # 비전공자 설명: 금융 데이터 서비스에 인터넷으로 요청해서 현재 환율을 받아옵니다.
    try:
        url = f"https://open.er-api.com/v6/latest/{base_currency}"
        resp = requests.get(url, timeout=10)
        data = resp.json()
        rate = data['rates'].get(target_currency)
        if rate:
            return json.dumps({"base": base_currency, "target": target_currency, "rate": rate})
        return json.dumps({"error": f"{target_currency} not found"})
    except Exception as e:
        return json.dumps({"error": str(e)})


# 함수 이름 -> 실제 함수 매핑
# 비전공자 설명: AI가 "get_weather 써야겠다"고 결정하면
#               이 딕셔너리를 통해 실제 파이썬 함수를 찾아서 실행합니다
available_functions = {
    'get_weather': get_weather,
    'calculate': calculate,
    'get_exchange_rate': get_exchange_rate
}

print("Function Calling 도구 및 함수 정의 완료")
print(f"  등록된 도구: {[t['function']['name'] for t in tools]}")


Function Calling 도구 및 함수 정의 완료
  등록된 도구: ['get_weather', 'calculate', 'get_exchange_rate']


---
### Function Calling ReAct 루프

텍스트 파싱 방식(예제 1)과 가장 큰 차이:
AI가 "Action: 도구이름[...]" 텍스트를 출력하는 대신,
**message.tool_calls 필드에 JSON으로 직접 넣어줍니다.** 코드가 파싱할 필요가 없습니다!


In [8]:
def run_function_calling_agent(question, tools, available_functions, max_steps=5):
    # ================================================================
    # 비전공자 설명 -- Function Calling 루프의 핵심 차이
    #
    # 텍스트 기반 방식 (예제 1):
    #   AI 출력 -> "Action: Calc[100+200]"  <- 개발자가 이 문장을 직접 해석
    #                                         <- 형식이 틀리면 오류 발생
    #
    # Function Calling 방식 (예제 2):
    #   AI 출력 -> message.tool_calls = [   <- 이미 구조화된 JSON
    #               {name: "calculate",
    #                arguments: '{"expression": "100+200"}'}
    #             ]                         <- 파싱 없이 바로 사용 가능!
    # ================================================================
    messages = [{'role': 'user', 'content': question}]

    print(f'질문: {question}')
    print('=' * 60)

    for step in range(max_steps):
        # (1) OpenAI API 호출 (tools 파라미터로 사용 가능한 도구 전달)
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            tools=tools,
            tool_choice='auto',  # AI가 도구 사용 여부를 자동 판단
            temperature=0
        )
        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            # (2) 도구 호출이 있을 경우 -- 각 도구 실행
            print(f'\n[Step {step+1}: 도구 호출]')
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                print(f'  -> {func_name}({func_args})')
                result = available_functions[func_name](**func_args)
                print(f'  <- 결과: {result}')

                # (3) 도구 실행 결과를 "tool" 역할로 대화에 추가
                # 비전공자 설명: AI가 다음 턴에 이 결과를 보고 다음 행동을 결정합니다
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    'content': result
                })
        else:
            # (4) 도구 호출이 없으면 = 최종 텍스트 답변
            print(f'\n[최종 답변]')
            print(message.content)
            return message.content

    return '최대 단계 도달'


---
### 실행 A — 날씨 비교 + 온도 차이 계산

실제 날씨 API를 호출하고, 온도 차이를 계산기로 계산합니다.

> 주목할 점: AI가 서울과 도쿄 날씨를 '동시에' 조회합니다. (Parallel Function Calling)
> 순서대로 하나씩 기다리지 않고 두 요청을 한 번에 처리합니다.


In [9]:
# ===================================================================
# 비전공자 설명 -- 이 질문은 3가지 도구가 필요합니다:
#   1. get_weather(Seoul)           -> 서울 날씨 조회
#   2. get_weather(Tokyo)           -> 도쿄 날씨 조회
#   3. calculate(서울온도 - 도쿄온도) -> 차이 계산
#
# 특이한 점: AI가 1번과 2번을 '동시에' 호출합니다. (Parallel Function Calling)
#           서로 독립적인 요청이면 AI가 자동으로 동시 처리합니다.
# ===================================================================

result_a = run_function_calling_agent(
    "서울과 도쿄의 현재 날씨를 비교해주세요. 두 도시의 온도 차이도 계산해주세요.",
    tools,
    available_functions
)


질문: 서울과 도쿄의 현재 날씨를 비교해주세요. 두 도시의 온도 차이도 계산해주세요.

[Step 1: 도구 호출]
  -> get_weather({'city': 'Seoul', 'unit': 'celsius'})
  <- 결과: {"city": "Seoul", "temperature": "26 degrees C", "description": "Sunny", "humidity": "45%"}
  -> get_weather({'city': 'Tokyo', 'unit': 'celsius'})
  <- 결과: {"city": "Tokyo", "temperature": "20 degrees C", "description": "Partly cloudy", "humidity": "73%"}

[Step 2: 도구 호출]
  -> calculate({'expression': '26 - 20'})
  <- 결과: {"expression": "26 - 20", "result": "6"}

[최종 답변]
현재 서울의 날씨는 다음과 같습니다:
- 온도: 26도 C
- 날씨: 맑음
- 습도: 45%

현재 도쿄의 날씨는 다음과 같습니다:
- 온도: 20도 C
- 날씨: 부분적으로 흐림
- 습도: 73%

두 도시의 온도 차이는 6도 C입니다.


---
### 실행 B — 실시간 환율 조회 + 환산 계산

실제 환율 API로 현재 환율을 가져오고, AI가 스스로 계산까지 처리합니다.


In [10]:
# ===================================================================
# 비전공자 설명 -- 이 예제가 강력한 이유:
#   1단계: get_exchange_rate(USD -> KRW) 로 실시간 환율 조회
#   2단계: AI가 조회된 환율을 바탕으로 220달러를 환산
#
#   -> 실제 인터넷 데이터를 가져와서 실용적인 답변을 만들어냅니다
#   -> AI의 고정된 지식이 아니라 지금 이 순간의 실제 환율입니다
# ===================================================================

result_b = run_function_calling_agent(
    "현재 1달러는 몇 원인지 조회하고, 220달러를 원화로 환산해주세요.",
    tools,
    available_functions
)


질문: 현재 1달러는 몇 원인지 조회하고, 220달러를 원화로 환산해주세요.

[Step 1: 도구 호출]
  -> get_exchange_rate({'base_currency': 'USD', 'target_currency': 'KRW'})
  <- 결과: {"base": "USD", "target": "KRW", "rate": 1532.415441}
  -> calculate({'expression': '220 * (환율)'})
  <- 결과: {"error": "name '\ud658\uc728' is not defined"}

[Step 2: 도구 호출]
  -> calculate({'expression': '220 * 1532.415441'})
  <- 결과: {"expression": "220 * 1532.415441", "result": "337131.39702000003"}

[최종 답변]
현재 1달러는 약 1,532.42원입니다. 따라서 220달러는 약 337,131.40원으로 환산됩니다.


---
## PART 6. 두 방식 비교 및 최종 정리

---

### 방식 비교 요약

| 항목 | 예제 1 (텍스트 파싱) | 예제 2 (Function Calling) |
|------|----------------------|---------------------------|
| AI 출력 형태 | 자유 텍스트 | 구조화된 JSON |
| 파싱 코드 필요 | 직접 작성 필요 | 불필요 |
| 병렬 도구 호출 | 불가 | 자동 지원 |
| 외부 API 연동 | 수동 구현 | 수동 구현 (동일) |
| 학습 난이도 | 구조 이해에 좋음 | 실무에 더 가까움 |
| 추천 용도 | ReAct 개념 학습 | 실제 서비스 개발 |

---

### 핵심 3줄 요약

```
1. ReAct = AI가 생각(Reason)하고 도구를 써서 행동(Act)하는 구조
2. Thought -> Action -> Observation 루프를 Finish까지 반복
3. 도구만 바꾸면 날씨, 환율, 계산, 검색 등 무엇이든 연결 가능
```

---

### 확장 아이디어 -- 도구를 바꾸면 에이전트의 능력이 바뀐다

```python
# 웹 검색 에이전트
agent.register_tool('Search', web_search, '인터넷에서 최신 정보를 검색합니다')

# 데이터 분석 에이전트
agent.register_tool('ReadCSV', read_csv,   'CSV 파일을 읽어 데이터를 반환합니다')
agent.register_tool('Chart',   make_chart, '데이터로 그래프를 생성합니다')

# 코드 실행 에이전트
agent.register_tool('RunCode', execute_python, '파이썬 코드를 실행하고 결과를 반환합니다')
```

ChatGPT의 플러그인 기능, 구글의 AI 검색, Claude의 도구 사용 기능 -- 모두 이 구조를 기반으로 합니다.

---
